In [1]:
import pandas as pd
import json
import unidecode
import re
import os



# --- 1. Sua Função de ID (Defina ela aqui pela última vez) ---
def generate_id_movie(title, release_year):
    title = str(title) if pd.notnull(title) else ""
    try:
        release_year = str(int(release_year))
    except:
        release_year = "0000"
    
    clear_title = title.lower().strip()
    clear_title = unidecode.unidecode(clear_title)
    clear_title = re.sub(r'[^a-z0-9]', '', clear_title)
    return f"{clear_title}_{release_year}"

# --- 2. Carregar os 3 JSONs ---
# Ajuste os caminhos conforme sua pasta
base_path = '../data' 

with open(f'{base_path}/movies_master_completo.json', 'r', encoding='utf-8') as f:
    data_master = pd.DataFrame(json.load(f))

with open(f'{base_path}/synopsis.json', 'r', encoding='utf-8') as f:
    data_synopsis = pd.DataFrame(json.load(f))

with open(f'{base_path}/critics.json', 'r', encoding='utf-8') as f:
    data_critics = pd.DataFrame(json.load(f))

# --- 3. Gerar IDs em TODOS os DataFrames ---
# Isso é crucial para garantir que o merge funcione perfeitamente
data_master['id_movie'] = data_master.apply(lambda x: generate_id_movie(x['title'], x['release_year']), axis=1)
data_synopsis['id_movie'] = data_synopsis.apply(lambda x: generate_id_movie(x['title'], x['release_year']), axis=1)
data_critics['id_movie'] = data_critics.apply(lambda x: generate_id_movie(x['title'], x['release_year']), axis=1)

# --- 4. O Grande Merge (Fusão) ---
# Começamos com o Mestre
df_final = data_master.copy()

# Juntamos com Sinopse (trazendo apenas o ID e a sinopse para não duplicar titulo/ano)
df_final = df_final.merge(
    data_synopsis[['id_movie', 'synopsis']], 
    on='id_movie', 
    how='left'
)

# Juntamos com Críticas (trazendo apenas ID e a lista de críticas)
df_final = df_final.merge(
    data_critics[['id_movie', 'critics']], 
    on='id_movie', 
    how='left'
)

# --- 5. Salvar o Dataset "Gold" ---
# Usar Parquet é o padrão da indústria. Requer instalar: pip install pyarrow fastparquet
output_file = f'{base_path}/dataset_gramado_completo.parquet'
df_final.to_parquet(output_file, index=False)

print(f"Dataset final salvo com sucesso! Dimensões: {df_final.shape}")
print(f"Colunas: {list(df_final.columns)}")

In [5]:


# --- 1. Sua Função de ID (Defina ela aqui pela última vez) ---
def generate_id_movie(title, release_year):
    title = str(title) if pd.notnull(title) else ""
    try:
        release_year = str(int(release_year))
    except:
        release_year = "0000"
    
    clear_title = title.lower().strip()
    clear_title = unidecode.unidecode(clear_title)
    clear_title = re.sub(r'[^a-z0-9]', '', clear_title)
    return f"{clear_title}_{release_year}"

# --- 2. Carregar os 3 JSONs ---
# Ajuste os caminhos conforme sua pasta
base_path = '../data' 

with open(f'{base_path}/movies_master_completo.json', 'r', encoding='utf-8') as f:
    data_master = pd.DataFrame(json.load(f))

with open(f'{base_path}/synopsis.json', 'r', encoding='utf-8') as f:
    data_synopsis = pd.DataFrame(json.load(f))

with open(f'{base_path}/critics.json', 'r', encoding='utf-8') as f:
    data_critics = pd.DataFrame(json.load(f))



# Para evitar o viés de filmes com nota alta mas poucos votos, será usada a MÉDIA BAYESIANA PONDERADA
# O dataset será dividido entre longas e curtas já que a diferença de votos entre os grupos é gritante
# W = (v/(v+m) * R) + (m/(v+m) * C)

def weighted_rating(df_input):
    groups = df_input.groupby('type')
    result = []

    for g_name, df_group in groups:
        # C = Média Global
        C = df_group['rating'].mean()
        # m = Mínimo de votos para ser considerado 25% mais votados
        m = df_group['vote_count'].quantile(0.25)

        print(f"Grupo {g_name}: Média Global (C)={C:.2f}, Corte de Votos (m)={m:.0f}")

        def weighted_formula(row):
            v = row['vote_count']
            R = row['rating']
            return (v/(v+m) * R) + (m/(v+m) * C)
        
        df_group = df_group.copy()
        df_group['score_sucesso'] = df_group.apply(weighted_formula, axis=1)
        result.append(df_group)

    return pd.concat(result)


data_master = weighted_rating(data_master)

data_master = data_master.drop(['rating', 'vote_count'], axis=1)

# --- 3. Gerar IDs em TODOS os DataFrames ---
# Isso é crucial para garantir que o merge funcione perfeitamente
data_master['id_movie'] = data_master.apply(lambda x: generate_id_movie(x['title'], x['release_year']), axis=1)
data_synopsis['id_movie'] = data_synopsis.apply(lambda x: generate_id_movie(x['title'], x['release_year']), axis=1)
data_critics['id_movie'] = data_critics.apply(lambda x: generate_id_movie(x['title'], x['release_year']), axis=1)




print(data_master.head())



Grupo cm: Média Global (C)=3.56, Corte de Votos (m)=41
Grupo lm: Média Global (C)=3.28, Corte de Votos (m)=184
         festival_name  festival_year  release_year       award_category  \
5  Festival de Gramado           2015          2015  [filme, fotografia]   
6  Festival de Gramado           2015          2015            [diretor]   
7  Festival de Gramado           2015          2015      [ator, roteiro]   
8  Festival de Gramado           2015          2015       [júri popular]   
9  Festival de Gramado           2015          2015    [júri da crítica]   

                                       title      directed by  duration type  \
5                                    O corpo   Lucas Cassales        17   cm   
6                           O Teto Sobre Nós    Bruno Carboni        22   cm   
7  Quando Parei de Me Preocupar com Canalhas     Tiago Vieira        15   cm   
8                                         Bá  Leandro Tadashi        14   cm   
9                       Dá Licen